# Bird Bioacoustics on Google Colab

Run the existing Python application on the Xeno-canto catalogue, one batch at a time.

Before starting:

1. Add `XENO_CANTO_API_KEY` in Colab's **Secrets** panel and enable notebook access. The key is read into memory and is never saved to Drive.

## Batch and inference settings

In [ ]:
BATCH_SIZE: int = 100
DEVICE: str = "CPU"  # @param ["CPU", "GPU:0"]
INFERENCE_WORKERS: int = 1
AUDIO_PRODUCERS: int = 2
INFERENCE_BATCH_SIZE: int = 16

## Runtime helpers


In [ ]:
from codecs import IncrementalDecoder, getincrementaldecoder
from contextlib import closing, suppress
from fcntl import LOCK_EX, LOCK_NB, LOCK_UN, flock
from os import environ, killpg
from os import read as read_pipe
from pathlib import Path
from shutil import copy2
from signal import SIGINT, SIGKILL
from sqlite3 import connect
from subprocess import (
    DEVNULL,
    PIPE,
    STDOUT,
    CalledProcessError,
    Popen,
    TimeoutExpired,
    run,
)
from sys import executable
from tempfile import TemporaryDirectory
from typing import IO, Final

from google.colab import drive, userdata
from IPython.display import Image, display


class ColabPaths:
    """Keep runtime and persistent paths in one place."""

    REPOSITORY: Final[str] = (
        "https://github.com/maximedrn/bird-bioacoustics.git"
    )
    BRANCH: Final[str] = "master"
    PYTHON: Final[str] = "3.13"
    API_KEY: Final[str] = "XENO_CANTO_API_KEY"
    PROJECT: Final[Path] = Path("/content/bird-bioacoustics")
    DRIVE: Final[Path] = Path("/content/drive")
    STORAGE: Final[Path] = DRIVE / "MyDrive" / PROJECT.name
    DATA: Final[Path] = PROJECT / "data"
    RESULTS: Final[Path] = DATA / "results"
    DATABASE: Final[Path] = DATA / "database.sqlite3"
    LOCK: Final[Path] = DATABASE.with_suffix(".lock")
    NOTEBOOK: Final[Path] = PROJECT / "notebook.ipynb"
    BACKUP: Final[Path] = STORAGE / DATABASE.name
    RESTORED: Final[Path] = DATA / ".colab-restored"
    MODELS: Final[Path] = DATA / "models"
    MODEL_READY: Final[Path] = DATA / ".colab-model-ready"
    ENVIRONMENT: Final[Path] = PROJECT / ".venv"
    INTERPRETER: Final[Path] = ENVIRONMENT / "bin" / "python"
    NVIDIA: Final[Path] = (
        ENVIRONMENT / "lib" / f"python{PYTHON}" / "site-packages" / "nvidia"
    )
    PENDING_SUFFIX: Final[str] = ".pending"


class ColabCommands:
    """Share the project environment and application command prefixes."""

    PROJECT: Final[tuple[str, ...]] = (
        "uv",
        "run",
        "--no-sync",
        "--project",
        str(ColabPaths.PROJECT),
    )
    APPLICATION: Final[tuple[str, ...]] = (
        *PROJECT,
        "app",
        "--root",
        str(ColabPaths.PROJECT),
    )
    GIT: Final[tuple[str, ...]] = ("git", "-C", str(ColabPaths.PROJECT))
    BATCH: Final[str] = "batch"
    MODELS: Final[str] = "models"
    BATCH_SIZE: Final[str] = "--batch-size"
    DEVICE: Final[str] = "--device"
    WORKERS: Final[str] = "--workers"
    PRODUCERS: Final[str] = "--producers"
    INFERENCE_BATCH_SIZE: Final[str] = "--inference-batch-size"
    REPORT: Final[str] = "report"
    STATUS: Final[str] = "status"


class ColabDevice:
    """Name the supported inference targets."""

    CPU: Final[str] = "CPU"
    GPU: Final[str] = "GPU:0"


class ColabMessage:
    """Centralize notebook status messages and operational errors."""

    MISSING_OUTPUT: Final[str] = "The command output stream is unavailable."
    INVALID_BATCH_SIZE: Final[str] = "BATCH_SIZE must be a positive integer."
    INVALID_DEVICE: Final[str] = "Choose CPU or GPU:0."
    SOURCE_CHANGES: Final[str] = (
        "Local source changes prevent an automatic update. "
        "Commit or save them before rerunning installation."
    )
    ACTIVE_BATCH: Final[str] = (
        "Stop the active batch before updating the project."
    )
    WRONG_BRANCH: Final[str] = "Select master before updating the project."
    STASH_CHANGED: Final[str] = (
        "The stash changed during the update; the artifact snapshot is kept."
    )
    MODEL_NOT_READY: Final[str] = (
        "Run Download and verify the model before processing audio."
    )
    CHECKPOINT_SAVED: Final[str] = "Checkpoint saved to {path}."
    REPORT_SAVED: Final[str] = "Results saved to {path}."


class ColabRuntime:
    """Share subprocess output and interruption settings."""

    OUTPUT_ENCODING: Final[str] = "utf-8"
    OUTPUT_ERRORS: Final[str] = "replace"
    OUTPUT_CHUNK_BYTES: Final[int] = 4096
    UNBUFFERED_VARIABLE: Final[str] = "PYTHONUNBUFFERED"
    UNBUFFERED_VALUE: Final[str] = "1"
    INTERRUPT_TIMEOUT: Final[float] = 30.0
    MODEL_CACHE_VARIABLE: Final[str] = "BIRDNET_APP_DATA"
    START_METHOD_VARIABLE: Final[str] = "BIRDNET_START_METHOD"
    START_METHOD: Final[str] = "spawn"
    ARTIFACT_STASH: Final[str] = "Colab runtime artifacts"
    STASH_REFERENCE: Final[str] = "refs/stash"
    TOP_STASH: Final[str] = "stash@{0}"
    ARTIFACT_PREFIX: Final[str] = "data/"
    LIBRARY_PATH_VARIABLE: Final[str] = "LD_LIBRARY_PATH"
    CPU_PACKAGE: Final[str] = "onnxruntime"
    GPU_PACKAGE: Final[str] = "onnxruntime-gpu"
    GPU_REQUIREMENT: Final[str] = "onnxruntime-gpu[cuda,cudnn]==1.26.0"


def interrupt_command(process: Popen[bytes]) -> None:
    """Stop the worker group before saving its committed checkpoint.

    :param process: Command started in its own process session.
    :type process: Popen[bytes]
    :return: None.
    :rtype: None
    """
    if process.poll() is not None:
        return
    with suppress(ProcessLookupError):
        killpg(process.pid, SIGINT)
    try:
        process.wait(timeout=ColabRuntime.INTERRUPT_TIMEOUT)
    except TimeoutExpired:
        with suppress(ProcessLookupError):
            killpg(process.pid, SIGKILL)
        process.wait()


def run_command(arguments: list[str]) -> None:
    """Stream stdout and stderr without Colab's zero-size pseudo-terminal.

    :param arguments: Executable and its individual arguments.
    :type arguments: list[str]
    :return: None.
    :rtype: None
    """
    environment: dict[str, str] = dict(environ)
    environment[ColabRuntime.MODEL_CACHE_VARIABLE] = str(ColabPaths.MODELS)
    environment[ColabRuntime.START_METHOD_VARIABLE] = ColabRuntime.START_METHOD
    libraries: list[str] = [
        str(path) for path in sorted(ColabPaths.NVIDIA.glob("*/lib"))
    ]
    existing_libraries: str = environment.get(
        ColabRuntime.LIBRARY_PATH_VARIABLE, ""
    )
    if existing_libraries:
        libraries.append(existing_libraries)
    if libraries:
        environment[ColabRuntime.LIBRARY_PATH_VARIABLE] = ":".join(libraries)
    environment[ColabRuntime.UNBUFFERED_VARIABLE] = (
        ColabRuntime.UNBUFFERED_VALUE
    )
    process: Popen[bytes] = Popen(
        arguments,
        stdin=DEVNULL,
        stdout=PIPE,
        stderr=STDOUT,
        bufsize=0,
        env=environment,
        start_new_session=True,
    )
    output: IO[bytes] | None = process.stdout
    decoder: IncrementalDecoder = getincrementaldecoder(
        ColabRuntime.OUTPUT_ENCODING
    )(errors=ColabRuntime.OUTPUT_ERRORS)
    try:
        if output is None:
            raise RuntimeError(ColabMessage.MISSING_OUTPUT)
        fragment: bytes
        while True:
            fragment = read_pipe(
                output.fileno(), ColabRuntime.OUTPUT_CHUNK_BYTES
            )
            if not fragment:
                break
            print(decoder.decode(fragment), end="", flush=True)
        print(decoder.decode(b"", final=True), end="", flush=True)
        exit_status: int = process.wait()
    except BaseException:
        interrupt_command(process)
        raise
    finally:
        if output is not None:
            output.close()
    if exit_status != 0:
        raise CalledProcessError(exit_status, arguments)


def publish_file(source: Path, destination: Path) -> None:
    """Replace an artifact only after its copy has completed.

    :param source: Closed local file to transfer.
    :type source: Path
    :param destination: Persistent destination file.
    :type destination: Path
    :return: None.
    :rtype: None
    """
    destination.parent.mkdir(parents=True, exist_ok=True)
    pending: Path = destination.with_name(
        destination.name + ColabPaths.PENDING_SUFFIX
    )
    copy2(source, pending)
    pending.replace(destination)


def save_checkpoint() -> None:
    """Back up committed measurements, including their write-ahead journal.

    :return: None.
    :rtype: None
    """
    with TemporaryDirectory() as directory:
        snapshot: Path = Path(directory) / ColabPaths.DATABASE.name
        with (
            closing(
                connect(
                    ColabPaths.DATABASE.as_uri() + "?mode=ro",
                    uri=True,
                )
            ) as source,
            closing(connect(snapshot)) as target,
        ):
            source.backup(target)
        publish_file(snapshot, ColabPaths.BACKUP)
    print(ColabMessage.CHECKPOINT_SAVED.format(path=ColabPaths.BACKUP))


def save_report() -> None:
    """Transfer the presentation notebook and cumulative report artifacts.

    :return: None.
    :rtype: None
    """
    publish_file(
        ColabPaths.NOTEBOOK,
        ColabPaths.STORAGE / ColabPaths.NOTEBOOK.name,
    )
    for artifact in sorted(ColabPaths.RESULTS.iterdir()):
        if not artifact.is_file():
            continue
        publish_file(
            artifact,
            ColabPaths.STORAGE / ColabPaths.RESULTS.name / artifact.name,
        )
    print(ColabMessage.REPORT_SAVED.format(path=ColabPaths.STORAGE))


def inference_arguments() -> list[str]:
    """Share resource settings between preparation and batch processing.

    :return: Inference arguments passed to the Python application.
    :rtype: list[str]
    """
    if DEVICE not in (ColabDevice.CPU, ColabDevice.GPU):
        raise ValueError(ColabMessage.INVALID_DEVICE)
    return [
        ColabCommands.DEVICE,
        DEVICE,
        ColabCommands.WORKERS,
        str(INFERENCE_WORKERS),
        ColabCommands.PRODUCERS,
        str(AUDIO_PRODUCERS),
        ColabCommands.INFERENCE_BATCH_SIZE,
        str(INFERENCE_BATCH_SIZE),
    ]


def capture_command(arguments: list[str]) -> str:
    """Read Git state while keeping command failures visible in the cell.

    :param arguments: Git executable and individual arguments.
    :type arguments: list[str]
    :return: Captured UTF-8 command output.
    :rtype: str
    """
    try:
        return run(
            arguments,
            stdout=PIPE,
            stderr=STDOUT,
            text=True,
            encoding=ColabRuntime.OUTPUT_ENCODING,
            check=True,
        ).stdout
    except CalledProcessError as error:
        print(error.output, end="", flush=True)
        raise


def update_project() -> None:
    """Pull the current master branch while keeping local experiment data.

    :return: None.
    :rtype: None
    """
    ColabPaths.DATA.mkdir(parents=True, exist_ok=True)
    with ColabPaths.LOCK.open("a") as lock:
        try:
            flock(lock, LOCK_EX | LOCK_NB)
        except BlockingIOError:
            raise RuntimeError(ColabMessage.ACTIVE_BATCH) from None
        try:
            pull_project()
        finally:
            flock(lock, LOCK_UN)


def pull_project() -> None:
    """Fast-forward source code while preserving existing generated files.

    :return: None.
    :rtype: None
    """
    branch: str = capture_command(
        [*ColabCommands.GIT, "symbolic-ref", "--short", "HEAD"]
    ).strip()
    if branch != ColabPaths.BRANCH:
        raise RuntimeError(ColabMessage.WRONG_BRANCH)
    changes: list[str] = [
        path
        for path in capture_command(
            [*ColabCommands.GIT, "diff", "--name-only", "-z", "HEAD"]
        ).split("\0")
        if path
    ]
    if any(
        not path.startswith(ColabRuntime.ARTIFACT_PREFIX)
        and path != ColabPaths.NOTEBOOK.name
        for path in changes
    ):
        raise RuntimeError(ColabMessage.SOURCE_CHANGES)
    artifacts: list[str] = [
        path
        for path in capture_command(
            [
                *ColabCommands.GIT,
                "ls-files",
                "-z",
                "--",
                ColabPaths.DATA.name,
                ColabPaths.NOTEBOOK.name,
            ]
        ).split("\0")
        if path
    ]
    with TemporaryDirectory() as directory:
        copies: list[tuple[Path, Path]] = []
        removed: list[Path] = []
        artifact: str
        for artifact in artifacts:
            original: Path = ColabPaths.PROJECT / artifact
            backup: Path = Path(directory) / artifact
            if not original.is_file():
                removed.append(original)
                continue
            backup.parent.mkdir(parents=True, exist_ok=True)
            copy2(original, backup)
            copies.append((backup, original))
        snapshot: str | None = None
        try:
            if changes:
                snapshot = stash_artifacts(changes)
            run_command(
                [
                    *ColabCommands.GIT,
                    "pull",
                    "--ff-only",
                    "origin",
                    ColabPaths.BRANCH,
                ]
            )
        finally:
            for source, target in copies:
                publish_file(source, target)
            absent: Path
            for absent in removed:
                absent.unlink(missing_ok=True)
            if snapshot is not None:
                drop_artifact_stash(snapshot)


def stash_artifacts(changes: list[str]) -> str:
    """Clean tracked artifact changes before Git merges updated source code.

    :param changes: Modified generated paths, already checked for safety.
    :type changes: list[str]
    :return: Created stash commit identifier.
    :rtype: str
    """
    run_command(
        [
            *ColabCommands.GIT,
            "stash",
            "push",
            "--message",
            ColabRuntime.ARTIFACT_STASH,
            "--",
            *changes,
        ]
    )
    return capture_command(
        [*ColabCommands.GIT, "rev-parse", ColabRuntime.STASH_REFERENCE]
    ).strip()


def drop_artifact_stash(snapshot: str) -> None:
    """Remove only this update's stash after local files are restored.

    :param snapshot: Stash identifier created before the pull.
    :type snapshot: str
    :return: None.
    :rtype: None
    """
    current_snapshot: str = capture_command(
        [*ColabCommands.GIT, "rev-parse", ColabRuntime.STASH_REFERENCE]
    ).strip()
    if current_snapshot != snapshot:
        raise RuntimeError(ColabMessage.STASH_CHANGED)
    run_command([*ColabCommands.GIT, "stash", "drop", ColabRuntime.TOP_STASH])

## Install or update the project

In [ ]:
run_command([executable, "-m", "pip", "install", "--quiet", "uv"])

if not ColabPaths.PROJECT.exists():
    run_command(
        [
            "git",
            "clone",
            "--branch",
            ColabPaths.BRANCH,
            "--depth",
            "1",
            ColabPaths.REPOSITORY,
            str(ColabPaths.PROJECT),
        ]
    )

else:
    update_project()

ColabPaths.MODEL_READY.unlink(missing_ok=True)
run_command(
    [
        "uv",
        "sync",
        "--project",
        str(ColabPaths.PROJECT),
        "--frozen",
        "--no-dev",
        "--python",
        ColabPaths.PYTHON,
    ]
)
run_command(
    [
        *ColabCommands.PROJECT,
        "playwright",
        "install",
        "--with-deps",
        "chromium",
    ]
)

## Configure CPU or CUDA inference


In [ ]:
if DEVICE not in (ColabDevice.CPU, ColabDevice.GPU):
    raise ValueError(ColabMessage.INVALID_DEVICE)

if DEVICE == ColabDevice.GPU:
    run_command(["nvidia-smi"])

ColabPaths.MODEL_READY.unlink(missing_ok=True)
run_command(
    [
        "uv",
        "pip",
        "uninstall",
        "--python",
        str(ColabPaths.INTERPRETER),
        ColabRuntime.CPU_PACKAGE,
        ColabRuntime.GPU_PACKAGE,
    ]
)

if DEVICE == ColabDevice.GPU:
    run_command(
        [
            "uv",
            "pip",
            "install",
            "--python",
            str(ColabPaths.INTERPRETER),
            ColabRuntime.GPU_REQUIREMENT,
        ]
    )
else:
    run_command(
        [
            "uv",
            "sync",
            "--project",
            str(ColabPaths.PROJECT),
            "--frozen",
            "--no-dev",
        ]
    )

## Download and verify the model


In [ ]:
ColabPaths.MODEL_READY.unlink(missing_ok=True)
run_command(
    [
        *ColabCommands.APPLICATION,
        ColabCommands.MODELS,
        *inference_arguments(),
    ]
)
ColabPaths.MODEL_READY.touch()

## Mount Drive and restore progress


In [ ]:
drive.mount(str(ColabPaths.DRIVE))
environ[ColabPaths.API_KEY] = userdata.get(ColabPaths.API_KEY)

ColabPaths.STORAGE.mkdir(parents=True, exist_ok=True)
ColabPaths.DATA.mkdir(parents=True, exist_ok=True)

if not ColabPaths.RESTORED.exists():
    if ColabPaths.BACKUP.is_file():
        copy2(ColabPaths.BACKUP, ColabPaths.DATABASE)
    ColabPaths.RESTORED.touch()

run_command([*ColabCommands.APPLICATION, ColabCommands.STATUS])
save_checkpoint()

## Run one batch


In [ ]:
if isinstance(BATCH_SIZE, bool) or BATCH_SIZE <= 0:
    raise ValueError(ColabMessage.INVALID_BATCH_SIZE)
if not ColabPaths.MODEL_READY.is_file():
    raise RuntimeError(ColabMessage.MODEL_NOT_READY)

try:
    run_command(
        [
            *ColabCommands.APPLICATION,
            ColabCommands.BATCH,
            ColabCommands.BATCH_SIZE,
            str(BATCH_SIZE),
            *inference_arguments(),
        ]
    )
finally:
    save_checkpoint()

## Generate cumulative results


In [ ]:
run_command([*ColabCommands.APPLICATION, ColabCommands.REPORT])
save_report()

In [ ]:
for figure in sorted(ColabPaths.RESULTS.glob("*.png")):
    display(Image(filename=str(figure)))

## Save progress manually

Use this cell to save the committed cursor before leaving a session, or to retry a backup if Drive was temporarily unavailable. It also displays the current experiment status.

In [ ]:
save_checkpoint()
run_command([*ColabCommands.APPLICATION, ColabCommands.STATUS])